# PII Masking 400k Results Analysis

This notebook analyzes `ner`/`langextract` alignment results and extracts all labels encountered in:

- `train_candidates.jsonl`
- `samples.jsonl`

It also identifies extra labels present in full `samples` that are missing from `train_candidates`.


In [ ]:
from pathlib import Path
import json
from collections import Counter

import pandas as pd
import numpy as np
import os

from datasets import load_dataset
import html
import unicodedata

import sys

BASE_DIR = Path('../../../resources/data/pii-masking-400k/qwen3:8b-260502_1504')
TRAIN_PATH = BASE_DIR / 'train_candidates.jsonl'
SAMPLES_PATH = BASE_DIR / 'samples.jsonl'

for p in [TRAIN_PATH, SAMPLES_PATH]:
    print(p, 'exists =', p.exists())


In [ ]:
ENTITY_FIELDS = ['ner_predictions', 'langextract_predictions', 'final_entities']


def iter_jsonl(path):
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)


def collect_labels(path, fields=ENTITY_FIELDS):
    all_counter = Counter()
    per_field = {field: Counter() for field in fields}
    row_count = 0

    for obj in iter_jsonl(path):
        row_count += 1
        for field in fields:
            for ent in obj.get(field, []) or []:
                label = ent.get('label')
                if label:
                    all_counter[label] += 1
                    per_field[field][label] += 1

    return {
        'rows': row_count,
        'all_counter': all_counter,
        'per_field': per_field,
        'labels': set(all_counter.keys()),
    }


In [ ]:
train_stats = collect_labels(TRAIN_PATH)
samples_stats = collect_labels(SAMPLES_PATH)

print('train rows:', train_stats['rows'])
print('samples rows:', samples_stats['rows'])
print('train unique labels:', len(train_stats['labels']))
print('samples unique labels:', len(samples_stats['labels']))


In [ ]:
train_labels = train_stats['labels']
samples_labels = samples_stats['labels']

extra_in_samples = sorted(samples_labels - train_labels)

print('Labels in train_candidates:')
print(sorted(train_labels))

print('Extra labels found in samples (candidate pool):')
print(extra_in_samples)

In [ ]:
def counter_to_df(counter, name='count'):
    if not counter:
        return pd.DataFrame(columns=['label', name])
    df = pd.DataFrame(counter.items(), columns=['label', name])
    return df.sort_values(name, ascending=False).reset_index(drop=True)

train_df = counter_to_df(train_stats['all_counter'], 'train_count')
samples_df = counter_to_df(samples_stats['all_counter'], 'samples_count')

summary = (
    train_df
    .merge(samples_df, on='label', how='outer')
    .fillna(0)
)
summary['train_count'] = summary['train_count'].astype(int)
summary['samples_count'] = summary['samples_count'].astype(int)
summary['is_extra_in_samples'] = summary['train_count'] == 0
summary = summary.sort_values(['is_extra_in_samples', 'samples_count'], ascending=[False, False]).reset_index(drop=True)

summary


In [ ]:
# Inspect where extra labels come from (ner vs langextract vs final_entities)
extra_labels = set(extra_in_samples)

rows = []
for field, counter in samples_stats['per_field'].items():
    for label in sorted(extra_labels):
        rows.append({
            'field': field,
            'label': label,
            'count': counter.get(label, 0),
        })

extra_breakdown = pd.DataFrame(rows).sort_values(['label', 'count'], ascending=[True, False]).reset_index(drop=True)
extra_breakdown


In [ ]:
from pathlib import Path

# Optional: export label reports for downstream use
reports_dir = Path(BASE_DIR) / 'reports'
reports_dir.mkdir(parents=True, exist_ok=True)

(summary.sort_values('samples_count', ascending=False)
 .to_csv(reports_dir / 'label_counts_train_vs_samples.csv', index=False))

pd.DataFrame({'train_labels': sorted(train_labels)}).to_csv(
    reports_dir / 'train_candidate_labels.csv', index=False
)

pd.DataFrame({'extra_labels_from_samples': extra_in_samples}).to_csv(
    reports_dir / 'extra_labels_from_samples.csv', index=False
)

extra_breakdown.to_csv(reports_dir / 'extra_labels_field_breakdown.csv', index=False)

print('Saved reports to', reports_dir)


## Label Histograms


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

def plot_label_hist(counter, title, top_n=30):
    df = counter_to_df(counter, 'count').head(top_n)
    plt.figure(figsize=(12, 6))
    sns.barplot(data=df, x='label', y='count', palette='viridis')
    plt.title(title)
    plt.xlabel('Label')
    plt.ylabel('Count')
    plt.xticks(rotation=60, ha='right')
    plt.tight_layout()

plot_label_hist(train_stats['all_counter'], 'Train Candidates Label Distribution')
plot_label_hist(samples_stats['all_counter'], 'Samples Label Distribution')


## BIO Token-Level DataFrames (NER vs LangExtract)


In [ ]:
import re

TOKEN_PATTERN = re.compile(r'\S+')


def find_label_for_token(token_start, token_end, entities):
    best_label = 'O'
    best_overlap = 0
    for ent in entities or []:
        start = ent.get('start_char')
        end = ent.get('end_char')
        label = ent.get('label')
        if start is None or end is None or not label:
            continue
        if start == token_start and end == token_end:
            return label
        overlap = max(0, min(token_end, end) - max(token_start, start))
        if overlap > best_overlap:
            best_overlap = overlap
            best_label = label
    return best_label


def to_bio_df(path, sample_limit=None):
    rows = []
    sample_count = 0

    for obj in iter_jsonl(path):
        sample_count += 1
        if sample_limit is not None and sample_count > sample_limit:
            break

        text = obj.get('text', '') or ''
        ner_entities = obj.get('ner_predictions', []) or []
        lx_entities = obj.get('langextract_predictions', []) or []
        paragraph_id = obj.get('paragraph_id')

        for m in TOKEN_PATTERN.finditer(text):
            token = m.group(0)
            token_start, token_end = m.start(), m.end()
            ner_label = find_label_for_token(token_start, token_end, ner_entities)
            lx_label = find_label_for_token(token_start, token_end, lx_entities)

            rows.append({
                'paragraph_id': paragraph_id,
                'token': token,
                'ner_pred': ner_label,
                'lx_pred': lx_label,
                'match': ner_label == lx_label,
                'label_defined': ner_label,
            })

    return pd.DataFrame(rows, columns=['paragraph_id', 'token', 'ner_pred', 'lx_pred', 'match', 'label_defined'])


def export_bio_txt(df, out_path, tag_col='ner_pred', token_col='token', paragraph_col='paragraph_id'):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    with out_path.open('w', encoding='utf-8') as f:
        for _, group in df.groupby(paragraph_col, dropna=False, sort=False):
            prev_label = 'O'
            for _, row in group.iterrows():
                token = str(row[token_col])
                label = str(row[tag_col]) if pd.notna(row[tag_col]) else 'O'

                if label == 'O':
                    bio_tag = 'O'
                elif label == prev_label:
                    bio_tag = f'I-{label}'
                else:
                    bio_tag = f'B-{label}'

                f.write(f"{token} {bio_tag}\n")
                prev_label = label
            f.write('\n')

    print('Saved BIO txt:', out_path)


In [ ]:
# Set sample_limit=None to process full files.
sample_limit = None

train_bio_df = to_bio_df(TRAIN_PATH, sample_limit=sample_limit)
samples_bio_df = to_bio_df(SAMPLES_PATH, sample_limit=sample_limit)

print('train_bio_df shape:', train_bio_df.shape)
print('samples_bio_df shape:', samples_bio_df.shape)


In [ ]:
samples_bio_df[samples_bio_df.paragraph_id == '24622']

### PER review

In [ ]:
import re

pattern = r'\b(rey|alc|cde|hipotecario|crédito|masculino|dip|reina|s.com|condesa|baron|baronesa|rei|sec|ing|ingeniero|ingeniera|príncipe|princesa|min|ministro|ministra|Prnc|profesor|prof|sen|senador|senadora|sargento|sgt)\b'

filtered_df = train_bio_df[
    (train_bio_df['ner_pred'] == 'PER') & 
    (train_bio_df['token'].str.contains(pattern, na=False, flags=re.IGNORECASE, regex=True))
].copy()

filtered_df


In [ ]:
train_bio_df.loc[filtered_df.index, 'label_defined'] = 'O'

In [ ]:
train_bio_df[(train_bio_df.paragraph_id == '25945')]

In [ ]:
train_bio_df[train_bio_df.ner_pred == 'PER'][1200:1250]

In [ ]:
df_path = os.path.join("../../../resources/outputs/pii-masking-400k/reviewing", "df_reviewed_27052601.parquet")

train_bio_df.to_parquet(df_path, index=True)

In [ ]:
train_bio_df = pd.read_parquet("../../../resources/outputs/pii-masking-400k/reviewing/df_reviewed_27052601.parquet")

In [ ]:
train_bio_df

### TEXTO_ANONIMIZAR review

In [ ]:
paragraphs_with_anon = train_bio_df.loc[
    train_bio_df['ner_pred'] == 'TEXTO_ANONIMIZAR', 'paragraph_id'
].unique()

result_df = train_bio_df[train_bio_df['paragraph_id'].isin(paragraphs_with_anon)].copy()

result_df[:50]

### CORREO_ELECTRONICO review

In [ ]:
email_regex_clean = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'

filtered_df = train_bio_df[
    (train_bio_df['ner_pred'] == 'CORREO_ELECTRONICO')
].copy()

invalid_emails_df = filtered_df[
    ~filtered_df['token'].str.contains(email_regex_clean, na=False, regex=True)
]

invalid_emails_df

In [ ]:
train_bio_df.loc[42620, 'label_defined'] = 'O'

In [ ]:
df_path = os.path.join("../../../resources/outputs/pii-masking-400k/reviewing", "df_reviewed_28052601.parquet")

train_bio_df.to_parquet(df_path, index=True)

### FECHA review

In [ ]:
filtered_df = train_bio_df[
    (train_bio_df['ner_pred'] == 'FECHA')
].copy()


filtered_df[50:100]

In [ ]:
dirty_date = (train_bio_df['ner_pred'] == 'FECHA') & (train_bio_df['token'].str.contains('T', na=False))

train_bio_df.loc[dirty_date, 'token'] = train_bio_df.loc[dirty_date, 'token'].str.split('T').str[0]

In [ ]:
df_path = os.path.join("../../../resources/outputs/pii-masking-400k/reviewing", "df_reviewed_28052602.parquet")

train_bio_df.to_parquet(df_path, index=True)

In [ ]:
train_bio_df = pd.read_parquet("../../../resources/outputs/pii-masking-400k/reviewing/df_reviewed_28052602.parquet")

### LINK review

In [ ]:
link_regex = r'https?://(?:www\.)?[-a-zA-Z0-9@:%._ South+~#=]{1,256}\.[a-zA-Z0-9()]{2,6}\b'

filtered_df = train_bio_df[
    (train_bio_df['ner_pred'] == 'LINK')
].copy()

invalid_links_df = filtered_df[
    ~filtered_df['token'].str.contains(link_regex, na=False, flags=re.IGNORECASE, regex=True)
]

invalid_links_df

filtered_df[450:500]

### LOC review

In [ ]:
filtered_df = train_bio_df[
    (train_bio_df['ner_pred'] == 'LOC')
].copy()

filtered_df[850:900]

### TELEFONO review

In [ ]:
filtered_df = train_bio_df[
    (train_bio_df['ner_pred'] == 'TELEFONO')
].copy()

filtered_df[50:100]

In [ ]:
train_bio_df[(train_bio_df.paragraph_id == '13274')]

### DIRECCION review

In [ ]:
filtered_df = train_bio_df[
    (train_bio_df['ner_pred'] == 'DIRECCION')
].copy()

filtered_df[700:750]

### EDAD review

In [ ]:
paragraphs_with_age = train_bio_df.loc[
    train_bio_df['ner_pred'] == 'EDAD', 'paragraph_id'
].unique()

result_df = train_bio_df[train_bio_df['paragraph_id'].isin(paragraphs_with_age)].copy()

result_df[2500:2550]

### BANCO review

In [ ]:
filtered_df = train_bio_df[
    (train_bio_df['ner_pred'] == 'BANCO')
].copy()

filtered_df[200:300]

### DNI review

In [ ]:
paragraphs_with_dni = train_bio_df.loc[
    train_bio_df['ner_pred'] == 'DNI', 'paragraph_id'
].unique()

result_df = train_bio_df[train_bio_df['paragraph_id'].isin(paragraphs_with_dni)].copy()

result_df[50:100]

In [ ]:
train_bio_df.loc[98862, 'label_defined'] = 'O'

In [ ]:
train_bio_df.loc[63889, 'token'] = '43768901'

In [ ]:
df_path = os.path.join("../../../resources/outputs/pii-masking-400k/reviewing", "df_reviewed_01062601.parquet")

train_bio_df.to_parquet(df_path, index=True)

In [ ]:
train_bio_df = pd.read_parquet("../../../resources/outputs/pii-masking-400k/reviewing/df_reviewed_01052601.parquet")

### NUM_EXPEDIENTE review

In [ ]:
paragraphs_with_exp = train_bio_df.loc[
    train_bio_df['ner_pred'] == 'NUM_EXPEDIENTE', 'paragraph_id'
].unique()

result_df = train_bio_df[train_bio_df['paragraph_id'].isin(paragraphs_with_exp)].copy()

result_df[100:150]

### PATENTE_DOMINIO review

In [ ]:
paragraphs_with_car = train_bio_df.loc[
    train_bio_df['ner_pred'] == 'PATENTE_DOMINIO', 'paragraph_id'
].unique()

result_df = train_bio_df[train_bio_df['paragraph_id'].isin(paragraphs_with_car)].copy()

result_df[150:200]

### CBU review

In [ ]:
paragraphs_with_cbu = train_bio_df.loc[
    train_bio_df['ner_pred'] == 'CBU', 'paragraph_id'
].unique()

result_df = train_bio_df[train_bio_df['paragraph_id'].isin(paragraphs_with_cbu)].copy()

result_df[100:150]

### IP review

In [ ]:
paragraphs_with_ip = train_bio_df.loc[
    train_bio_df['ner_pred'] == 'IP', 'paragraph_id'
].unique()

result_df = train_bio_df[train_bio_df['paragraph_id'].isin(paragraphs_with_ip)].copy()

result_df

### MARCA_AUTOMOVIL review

In [ ]:
paragraphs_with_carb = train_bio_df.loc[
    train_bio_df['ner_pred'] == 'MARCA_AUTOMOVIL', 'paragraph_id'
].unique()

result_df = train_bio_df[train_bio_df['paragraph_id'].isin(paragraphs_with_carb)].copy()

result_df

In [ ]:
train_bio_df.loc[46759, 'token'] = 'HONDA'

### Now we will compare the labels from the original Huggingface dataset with our alignment experiment predictions to see if there is an span match in the samples that aren't part of the train candidates. This will give us more candidates that are useful for the training of our NER from Flair.

In [ ]:
train_paragraphs = train_bio_df['paragraph_id'].unique()

span_train_bio_df = samples_bio_df[~samples_bio_df['paragraph_id'].isin(train_paragraphs)].copy()

In [ ]:
TAG_RE = re.compile(r"(?is)<[^>]+>")
SCRIPT_STYLE_RE = re.compile(r"(?is)<(script|style)[^>]*>.*?</\1>")
BREAK_TAG_RE = re.compile(r"(?is)<\s*(br|/p|/div|/li|/tr)\s*/?>")


def preprocess_hf_text(text: str, collapse_lines: bool = True) -> str:
    value = str(text or "")
    value = html.unescape(value)
    value = SCRIPT_STYLE_RE.sub(" ", value)
    value = BREAK_TAG_RE.sub("\n", value)
    value = TAG_RE.sub(" ", value)
    value = value.replace("\r\n", "\n").replace("\r", "\n")
    value = "".join(
        ch for ch in value
        if ch in {"\n", "\t", " "} or not unicodedata.category(ch).startswith("C")
    )
    value = re.sub(r"[ \t\f\v]+", " ", value)
    value = re.sub(r"\n{2,}", "\n", value)
    value = "\n".join(line.strip() for line in value.split("\n"))
    value = "\n".join(line for line in value.split("\n") if line)
    value = value.strip()
    if collapse_lines:
        value = re.sub(r"\s+", " ", value).strip()
    return value


def _as_mask_items(value):
    if isinstance(value, list):
        return value
    if isinstance(value, dict):
        return [value]
    return []


def _row_privacy_masks(row):
    for key in ["privacy_mask", "privacy_masks", "pii_spans", "spans", "span_labels"]:
        masks = _as_mask_items(row.get(key))
        if masks:
            return masks
    return []


def _mask_label(mask):
    for key in ["label", "entity", "entity_type", "type", "class"]:
        value = mask.get(key) if isinstance(mask, dict) else None
        if value:
            return str(value)
    return "HF_TAG"


def _mask_text(mask):
    for key in ["value", "text", "source_text", "span", "original_text"]:
        value = mask.get(key) if isinstance(mask, dict) else None
        if value:
            return str(value)
    return ""


def _mask_offsets(mask):
    if not isinstance(mask, dict):
        return None, None
    start = mask.get("start") or mask.get("start_char") or mask.get("begin")
    end = mask.get("end") or mask.get("end_char") or mask.get("stop")
    try:
        return int(start), int(end)
    except (TypeError, ValueError):
        return None, None


def _find_clean_span(clean_text, raw_text, mask):
    start, end = _mask_offsets(mask)
    pieces = []

    if start is not None and end is not None and 0 <= start < end <= len(raw_text):
        pieces.append(raw_text[start:end])

    mask_value = _mask_text(mask)
    if mask_value:
        pieces.append(mask_value)

    for piece in pieces:
        clean_piece = preprocess_hf_text(piece, collapse_lines=True)
        if not clean_piece:
            continue
        found = clean_text.find(clean_piece)
        if found >= 0:
            return found, found + len(clean_piece)

    return None, None


def _label_for_hf_token(token_start, token_end, spans):
    best_label = "O"
    best_overlap = 0
    for span in spans:
        overlap = max(0, min(token_end, span["end_char"]) - max(token_start, span["start_char"]))
        if overlap > best_overlap:
            best_overlap = overlap
            best_label = span["label"]
    return best_label


def build_hf_original_span_df(sample_limit=None):
    ds = load_dataset("ai4privacy/pii-masking-400k")
    train_rows = ds["train"].filter(lambda x: x["language"] == "es")

    rows = []
    written = 0
    for idx, row in enumerate(train_rows):
        if sample_limit is not None and written >= sample_limit:
            break

        raw_text = row.get("source_text") or ""
        clean_text = preprocess_hf_text(raw_text, collapse_lines=True)
        if not clean_text:
            continue

        paragraph_id = str(idx)
        clean_spans = []
        for mask in _row_privacy_masks(row):
            start, end = _find_clean_span(clean_text, raw_text, mask)
            if start is None or end is None:
                continue
            clean_spans.append({
                "start_char": start,
                "end_char": end,
                "label": _mask_label(mask),
            })

        for token_idx, m in enumerate(TOKEN_PATTERN.finditer(clean_text)):
            hf_label = _label_for_hf_token(m.start(), m.end(), clean_spans)
            rows.append({
                "paragraph_id": paragraph_id,
                "token_idx": token_idx,
                "hf_original_span": hf_label,
                "hf_has_tag": hf_label != "O",
            })

        written += 1

    return pd.DataFrame(rows)


def add_hf_binary_span_matches(span_df, hf_span_df):
    result = span_df.copy()
    result["token_idx"] = result.groupby("paragraph_id", sort=False).cumcount()

    result = result.merge(
        hf_span_df,
        on=["paragraph_id", "token_idx"],
        how="left",
    )

    result["hf_original_span"] = result["hf_original_span"].fillna("O")
    result["hf_has_tag"] = result["hf_has_tag"].fillna(False).astype(bool)
    result["ner_has_tag"] = result["ner_pred"].ne("O")
    result["lx_has_tag"] = result["lx_pred"].ne("O")
    result["any_pred_has_tag"] = result["ner_has_tag"] | result["lx_has_tag"]
    result["ner_binary_match"] = result["ner_has_tag"] == result["hf_has_tag"]
    result["lx_binary_match"] = result["lx_has_tag"] == result["hf_has_tag"]
    result["any_pred_binary_match"] = result["any_pred_has_tag"] == result["hf_has_tag"]

    return result.drop(columns=["token_idx", "hf_has_tag", "ner_has_tag", "lx_has_tag", "any_pred_has_tag", "ner_binary_match", "lx_binary_match"])


hf_original_span_df = build_hf_original_span_df()
span_train_bio_df = add_hf_binary_span_matches(span_train_bio_df, hf_original_span_df)

print("HF original span rows:", hf_original_span_df.shape)
print("span_train_bio_df rows:", span_train_bio_df.shape)
print("Any prediction binary match rate:", round(span_train_bio_df["any_pred_binary_match"].mean() * 100, 2), "%")


In [ ]:
span_train_bio_df[:50]

In [ ]:
binary_match_bio_df = span_train_bio_df.groupby('paragraph_id').filter(
    lambda x: x['any_pred_binary_match'].all()
).copy()

In [ ]:
binary_match_bio_df[50:100]

In [ ]:
if 'label_defined' and 'match' and 'any_pred_binary_match' in binary_match_bio_df.columns:
    binary_match_bio_df = binary_match_bio_df.drop(columns=['label_defined', 'match', 'any_pred_binary_match'])

conditions = [
    binary_match_bio_df['ner_pred'] == binary_match_bio_df['lx_pred'],
    binary_match_bio_df['ner_pred'] != binary_match_bio_df['lx_pred']
]

values = [
    binary_match_bio_df['ner_pred'],
    binary_match_bio_df['hf_original_span']
]

binary_match_bio_df['label_defined'] = np.select(conditions, values, default='O')

In [ ]:
binary_match_bio_df[:50]

In [ ]:
F_SPAN_TO_LABEL = {
    'STREET': 'DIRECCION',
    'CITY': 'LOC',
    'USERNAME': 'USUARIX',
    'ACCOUNTNUM': 'NUM_CAJA_AHORRO',
    'GIVENNAME': 'PER',
    'DATEOFBIRTH': 'FECHA',
    'CREDITCARDNUMBER': 'TEXTO_ANONIMIZAR',
    'PASSWORD': 'TEXTO_ANONIMIZAR',
    'SURNAME': 'PER',
    'EMAIL': 'CORREO_ELECTRONICO',
    'TELEPHONENUM': 'TELEFONO',
    'BUILDINGNUM': 'DIRECCION',
}

f_span_col = 'f_span' if 'f_span' in binary_match_bio_df.columns else 'hf_original_span'

mapped_labels = binary_match_bio_df[f_span_col].map(F_SPAN_TO_LABEL)
binary_match_bio_df['label_translate'] = mapped_labels.fillna(binary_match_bio_df['label_defined'])

label_translate_counts_df = (
    binary_match_bio_df['label_translate']
    .fillna('O')
    .value_counts()
    .rename_axis('label_translate')
    .reset_index(name='count')
)

label_translate_counts_df

In [ ]:
df_path = os.path.join("../../../resources/outputs/pii-masking-400k/reviewing", "span_df_reviewed_01062601.parquet")

binary_match_bio_df.to_parquet(df_path, index=True)

### Histogram of `label_defined` in `binary_match_bio_df`


In [ ]:
label_defined_counter = Counter(binary_match_bio_df['label_translate'].fillna('O'))
label_defined_counts_df = (
    pd.DataFrame(label_defined_counter.items(), columns=['label_translate', 'count'])
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
)

label_defined_counts_df

In [ ]:
plot_df = label_defined_counts_df.copy()

# Keep O visible in the counts table above, but remove it from the label histogram by default.
plot_df = plot_df[plot_df['label_translate'] != 'O']

plt.figure(figsize=(12, 6))
sns.barplot(data=plot_df, x='label_translate', y='count', palette='viridis')
plt.title('binary_match_bio_df label_translate distribution')
plt.xlabel('label_translate')
plt.ylabel('Count')
plt.xticks(rotation=60, ha='right')
plt.tight_layout()


In [ ]:
binary_match_bio_df['left_defined'] = (
    (binary_match_bio_df['label_translate'] != 'O') & 
    (binary_match_bio_df['label_translate'] == binary_match_bio_df['hf_original_span'])
)

In [ ]:
df_path = os.path.join("../../../resources/outputs/pii-masking-400k/reviewing", "span_df_reviewed_01062602.parquet")

binary_match_bio_df.to_parquet(df_path, index=True)

In [ ]:
binary_match_bio_df['label_translate'].unique()

### FECHA review

In [ ]:
filtered_df = binary_match_bio_df[
    (binary_match_bio_df['label_translate'] == 'FECHA')
].copy()


filtered_df[50:100]

In [ ]:
dirty_date = (binary_match_bio_df['ner_pred'] == 'FECHA') & (binary_match_bio_df['token'].str.contains('T', na=False))

binary_match_bio_df.loc[dirty_date, 'token'] = binary_match_bio_df.loc[dirty_date, 'token'].str.split('T').str[0]

In [ ]:
df_path = os.path.join("../../../resources/outputs/pii-masking-400k/reviewing", "span_df_reviewed_02062601.parquet")

binary_match_bio_df.to_parquet(df_path, index=True)

### PER review

In [ ]:
filtered_df = binary_match_bio_df[
    (binary_match_bio_df['label_translate'] == 'PER')
].copy()


filtered_df[:50]

### SOCIALNUM review

In [ ]:
conflict_paragraphs_soc = binary_match_bio_df.loc[
    (binary_match_bio_df['left_defined'] == True) &
    (binary_match_bio_df['label_translate'] == 'SOCIALNUM'),
    'paragraph_id'
].unique()


In [ ]:
filtered_df = binary_match_bio_df.loc[binary_match_bio_df['paragraph_id'].isin(conflict_paragraphs_soc)].copy()

In [ ]:
filtered_df[:50]

In [ ]:
base_condition = (
    (binary_match_bio_df['left_defined'] == True) & 
    (binary_match_bio_df['label_translate'] == 'SOCIALNUM')
)

extended_mask = base_condition.copy()
for i in range(1, 5):
    extended_mask |= base_condition.shift(-i, fill_value=False)

result_df = binary_match_bio_df[extended_mask]

In [ ]:
result_df[1200:1250]

In [ ]:
binary_match_bio_df.loc[
    binary_match_bio_df['label_translate'] == 'SOCIALNUM', 'label_translate'
] = 'TEXTO_ANONIMIZAR'

In [ ]:
df_path = os.path.join("../../../resources/outputs/pii-masking-400k/reviewing", "span_df_reviewed_02062602.parquet")

binary_match_bio_df.to_parquet(df_path, index=True)

### TAXNUM review

In [ ]:
base_condition = (
    (binary_match_bio_df['left_defined'] == True) & 
    (binary_match_bio_df['label_translate'] == 'TAXNUM')
)

extended_mask = base_condition.copy()
for i in range(1, 5):
    extended_mask |= base_condition.shift(-i, fill_value=False)

result_df = binary_match_bio_df[extended_mask]

In [ ]:
result_df[600:650]

In [ ]:
binary_match_bio_df.loc[
    binary_match_bio_df['label_translate'] == 'TAXNUM', 'label_translate'
] = 'CUIT_CUIL'

### ZIPCODE review

In [ ]:
base_condition = (
    (binary_match_bio_df['left_defined'] == True) & 
    (binary_match_bio_df['label_translate'] == 'ZIPCODE')
)

extended_mask = base_condition.copy()
for i in range(1, 4):
    extended_mask |= base_condition.shift(-i, fill_value=False)

result_df = binary_match_bio_df[extended_mask]

In [ ]:
result_df[900:950]

In [ ]:
binary_match_bio_df.loc[
    binary_match_bio_df['label_translate'] == 'ZIPCODE', 'label_translate'
] = 'TEXTO_ANONIMIZAR'

In [ ]:
df_path = os.path.join("../../../resources/outputs/pii-masking-400k/reviewing", "span_df_reviewed_02062604.parquet")

binary_match_bio_df.to_parquet(df_path, index=True)

### DRIVERLICENSENUM review

In [ ]:
base_condition = (
    (binary_match_bio_df['left_defined'] == True) & 
    (binary_match_bio_df['label_translate'] == 'DRIVERLICENSENUM')
)

extended_mask = base_condition.copy()
for i in range(1, 4):
    extended_mask |= base_condition.shift(-i, fill_value=False)

result_df = binary_match_bio_df[extended_mask]

In [ ]:
result_df[150:200]

In [ ]:
binary_match_bio_df.loc[
    binary_match_bio_df['label_translate'] == 'DRIVERLICENSENUM', 'label_translate'
] = 'TEXTO_ANONIMIZAR'

In [ ]:
df_path = os.path.join("../../../resources/outputs/pii-masking-400k/reviewing", "span_df_reviewed_02062605.parquet")

binary_match_bio_df.to_parquet(df_path, index=True)

In [ ]:
binary_match_bio_df = pd.read_parquet("../../../resources/outputs/pii-masking-400k/reviewing/span_df_reviewed_02062605.parquet")

### IDCARDNUM review

In [ ]:
new_order= [
    'paragraph_id', 
    'token', 
    'label_translate',  # <- La movemos acá
    'ner_pred', 
    'lx_pred', 
    'hf_original_span', 
    'label_defined', 
    'left_defined'
]

binary_match_bio_df = binary_match_bio_df[new_order]

In [ ]:
conflict_paragraphs_id = binary_match_bio_df.loc[
    (binary_match_bio_df['left_defined'] == True) &
    (binary_match_bio_df['label_translate'] == 'IDCARDNUM'),
    'paragraph_id'
].unique()

filtered_df = binary_match_bio_df.loc[binary_match_bio_df['paragraph_id'].isin(conflict_paragraphs_id)].copy()


In [ ]:
filtered_df[6450:6500]

In [ ]:
binary_match_bio_df.loc[420144, 'label_translate'] ='IP'

In [ ]:
binary_match_bio_df.loc[420144, 'token'] = '162.673.857.900'

In [ ]:
df_path = os.path.join("../../../resources/outputs/pii-masking-400k/reviewing", "span_df_reviewed_03062602.parquet")

binary_match_bio_df.to_parquet(df_path, index=True)

In [ ]:
binary_match_bio_df['paragraph_id'].nunique()

In [ ]:
project_root = Path.cwd()
while not (project_root / 'aymurai').exists() and project_root != project_root.parent:
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from aymurai.experiments.data_augmentation.anonymizer_entities import faker

In [ ]:
idcardnum_mask = binary_match_bio_df['label_translate'].eq('IDCARDNUM')
unique_idcard_tokens = binary_match_bio_df.loc[idcardnum_mask, 'token'].dropna().astype(str).unique()

generated_dnis = set()

def generate_unique_dni():
    while True:
        dni = faker.dni()
        if dni not in generated_dnis:
            generated_dnis.add(dni)
            return dni

idcardnum_to_dni = {
    original_token: generate_unique_dni()
    for original_token in unique_idcard_tokens
}

binary_match_bio_df.loc[idcardnum_mask, 'token_original_idcardnum'] = (
    binary_match_bio_df.loc[idcardnum_mask, 'token'].astype(str)
)
binary_match_bio_df.loc[idcardnum_mask, 'token'] = (
    binary_match_bio_df.loc[idcardnum_mask, 'token'].astype(str).map(idcardnum_to_dni)
)
binary_match_bio_df.loc[idcardnum_mask, 'label_translate'] = 'DNI'

idcardnum_dni_map_df = (
    pd.DataFrame(idcardnum_to_dni.items(), columns=['original_token', 'generated_dni'])
    .sort_values('original_token')
    .reset_index(drop=True)
)

print('IDCARDNUM tokens replaced:', int(idcardnum_mask.sum()))
print('DNIs unique generated:', len(idcardnum_to_dni))
idcardnum_dni_map_df


In [ ]:
df_path = os.path.join("../../../resources/outputs/pii-masking-400k/reviewing", "span_df_reviewed_03062603.parquet")

binary_match_bio_df.to_parquet(df_path, index=True)

In [ ]:
binary_match_bio_df = pd.read_parquet("../../../resources/outputs/pii-masking-400k/reviewing/span_df_reviewed_03062603.parquet")

In [ ]:
binary_match_bio_df.drop(columns=['token_original_idcardnum', 'left_defined', 'label_defined'], inplace=True)

In [ ]:
train_bio_df = pd.read_parquet("../../../resources/outputs/pii-masking-400k/reviewing/df_reviewed_01062601.parquet")

In [ ]:
binary_match_bio_df['label_defined'] = binary_match_bio_df['label_translate']

final_columns = ['paragraph_id', 'token', 'ner_pred', 'lx_pred', 'label_defined']
binary_match_bio_df = binary_match_bio_df[final_columns].copy()

In [ ]:
binary_match_bio_df['label_defined'].unique()

In [ ]:
binary_match_bio_df[:50]

In [ ]:
train_bio_df[:50]

In [ ]:
samples_bio_df.drop(columns=['match'], inplace=True, errors='ignore')

In [ ]:
samples_bio_df[:50]

In [ ]:
df_path = os.path.join("../../../resources/outputs/pii-masking-400k/reviewing", "span_df_reviewed_04062601.parquet")

binary_match_bio_df.to_parquet(df_path, index=True)

In [ ]:
df_path = os.path.join("../../../resources/outputs/pii-masking-400k/reviewing", "df_reviewed_04062601.parquet")

train_bio_df.to_parquet(df_path, index=True)

In [ ]:
combined_df = pd.concat([train_bio_df, binary_match_bio_df], ignore_index=True)

final_train_bio_df = combined_df.sort_values(by='paragraph_id', kind='stable').reset_index(drop=True)

In [ ]:
final_train_bio_df['paragraph_id'].nunique()

In [ ]:
df_path = os.path.join("../../../resources/outputs/pii-masking-400k/reviewing", "train_candidates_reviewed_04062601.parquet")

final_train_bio_df.to_parquet(df_path, index=True)

In [ ]:
defined_paragraphs = final_train_bio_df['paragraph_id'].unique()

left_defined_bio_df = samples_bio_df[~samples_bio_df['paragraph_id'].isin(defined_paragraphs)].copy()

In [ ]:
left_defined_bio_df[50:100]

In [ ]:
print('Train token-level match rate:', round(train_bio_df['match'].mean() * 100, 2), '%')
print('Samples token-level match rate:', round(samples_bio_df['match'].mean() * 100, 2), '%')

In [ ]:
OUTPUT_DIR = '../../../resources/outputs/pii-masking-400k/reports'  # string or Path
reports_dir = Path(OUTPUT_DIR)
reports_dir.mkdir(parents=True, exist_ok=True)

In [ ]:

# CSV exports
train_bio_out_csv = reports_dir / 'train_candidates_bio_tokens.csv'
samples_bio_out_csv = reports_dir / 'samples_bio_tokens.csv'
train_bio_df.to_csv(train_bio_out_csv, index=False)
samples_bio_df.to_csv(samples_bio_out_csv, index=False)
print('Saved:', train_bio_out_csv)
print('Saved:', samples_bio_out_csv)

In [ ]:
train_bio_out_txt = reports_dir / 'train_candidates_bio.txt'


export_bio_txt(final_train_bio_df, train_bio_out_txt, tag_col='label_defined')